<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/03_rev_and_uncertainty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 3: Representative Elementary Volume (REV) and Uncertainty

When you compute the tortuosity of a single crop from a micro-CT scan, the result depends on where and how large that crop is. A natural question is: *at what sample size do the computed properties stabilise?* This is the concept of the **Representative Elementary Volume (REV)**.

In this tutorial we take a statistical approach — computing the tortuosity of many random sub-volumes at different sizes — to identify the REV and quantify the associated uncertainty.

**What you will learn:**
1. Extract random sub-volumes of varying sizes from a 3D image.
2. Solve the transport problem for each crop (over 100 simulations).
3. Plot a violin-style REV diagram showing the full distribution at each scale.
4. Use the Coefficient of Variation (CV) to define the REV threshold.
5. Examine the physical correlation between local porosity and local tortuosity.

In [ ]:
# Install OpenImpala and utilities (takes < 5 seconds)
!pip install -q openimpala tifffile matplotlib

In [ ]:
import os
import time
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded.")

# Keep AMReX and MPI alive for the entire notebook.
global_session = oi.Session()
global_session.__enter__()

# Download the sample TIFF so the notebook works on Colab.
data_url = "https://raw.githubusercontent.com/BASE-Laboratory/OpenImpala/master/data/SampleData_2Phase_stack_3d_1bit.tif"
data_path = "SampleData_2Phase_stack_3d_1bit.tif"

if not os.path.exists(data_path):
    print("Downloading sample data...")
    urllib.request.urlretrieve(data_url, data_path)
    print("Download complete.")

# Load the scan into memory
microstructure = tifffile.imread(data_path).astype(np.int32)
print(f"Base microstructure loaded: {microstructure.shape}")

## 1. Setting Up the Statistical Sampling

If you pick a small sub-volume (say $16^3$), the tortuosity will vary substantially depending on whether you happen to sample a region with large pores or dense solid. As the sub-volume size increases, these fluctuations should average out and the computed properties should converge.

We define a set of window sizes. For each size, we randomly crop the microstructure 20 times and compute both the local porosity and tortuosity for every crop.

In [ ]:
window_sizes = [16, 24, 32, 40, 48, 64, 80]
n_samples_per_size = 20  # 140 simulations in total

rev_results = {w: {'tau': [], 'vf': []} for w in window_sizes}

print(f"Running {len(window_sizes) * n_samples_per_size} simulations...")
t_start = time.time()
total_solves = 0

for w in window_sizes:
    max_start_idx = microstructure.shape[0] - w

    for _ in range(n_samples_per_size):
        # 1. Random crop origin
        z = np.random.randint(0, max_start_idx + 1)
        y = np.random.randint(0, max_start_idx + 1)
        x = np.random.randint(0, max_start_idx + 1)

        # 2. Extract the sub-volume
        crop = microstructure[z:z+w, y:y+w, x:x+w]

        # 3. Local porosity
        vf = oi.volume_fraction(crop, phase=1).fraction

        # 4. Percolation check
        perc = oi.percolation_check(crop, phase=1, direction="z")

        # 5. Solve if percolating
        if perc.percolates:
            res = oi.tortuosity(crop, phase=1, direction="z", solver="flexgmres")
            rev_results[w]['tau'].append(res.tortuosity)
            rev_results[w]['vf'].append(vf)
            total_solves += 1

t_end = time.time()
print(f"\nCompleted {total_solves} solves in {t_end - t_start:.2f}s.")

## 2. REV Violin Diagram

A mean and standard deviation assume the data are normally distributed. A **violin plot** shows the full probability density at each scale, which is more informative — particularly at small sizes where the distribution can be strongly skewed (long tails of high-tortuosity bottleneck regions).

In [ ]:
means = []
stds = []
valid_sizes = []
plot_data = []

# Extract data, ignoring sizes where nothing percolated
for w in window_sizes:
    taus = rev_results[w]['tau']
    if len(taus) > 0:
        means.append(np.mean(taus))
        stds.append(np.std(taus))
        valid_sizes.append(w)
        plot_data.append(taus)

means = np.array(means)
stds = np.array(stds)
valid_sizes = np.array(valid_sizes)

fig, ax = plt.subplots(figsize=(9, 5), dpi=150)

# Render the Violin Plot
parts = ax.violinplot(plot_data, positions=valid_sizes, widths=5,
                      showmeans=False, showextrema=False)

# Style the violins
for pc in parts['bodies']:
    pc.set_facecolor('#1f77b4')
    pc.set_edgecolor('black')
    pc.set_alpha(0.5)

# Plot the Mean trend line
ax.plot(valid_sizes, means, color='#d62728', linewidth=2.5, marker='o', markersize=6, label='Mean Tortuosity')

ax.set_xlabel("Window Size (Voxels)", fontsize=12, fontweight='bold')
ax.set_ylabel(r"Tortuosity Factor ($\tau$)", fontsize=12, fontweight='bold')
ax.set_title("Probability Density of Microstructural Transport (REV)", fontsize=14, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend()

plt.tight_layout()
plt.show()

## 3. Quantifying Uncertainty with the Coefficient of Variation

To determine whether a given sample size is representative, we use the **Coefficient of Variation**: $\mathrm{CV} = \sigma / \mu \times 100\%$. A common practical threshold is CV < 5%.

The right-hand plot below examines *why* the variance is large at small scales: it shows the correlation between local porosity and local tortuosity for the $32^3$ sub-volumes. In a small crop, the porosity itself fluctuates, and tortuosity responds accordingly.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

# --- SUBPLOT 1: Coefficient of Variation ---
cvs = (stds / means) * 100
ax1.plot(valid_sizes, cvs, color='#2ca02c', linewidth=2.5, marker='s', markersize=7)
ax1.axhline(y=5.0, color='r', linestyle='--', linewidth=2, label='5% Threshold (REV Limit)')

ax1.set_xlabel("Window Size (Voxels)", fontweight='bold')
ax1.set_ylabel("Coefficient of Variation (%)", fontweight='bold')
ax1.set_title("Defining the REV Threshold", fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- SUBPLOT 2: The Physical Origin of Variance ---
# Let's look at the sub-volumes from the W=32 window size
w_small = 32
local_vfs = np.array(rev_results[w_small]['vf']) * 100  # Convert to percentage
local_taus = np.array(rev_results[w_small]['tau'])

ax2.scatter(local_vfs, local_taus, color='#ff7f0e', edgecolor='black', s=60, alpha=0.8)

# Add a linear trendline
if len(local_vfs) > 1:
    z = np.polyfit(local_vfs, local_taus, 1)
    p = np.poly1d(z)
    ax2.plot(local_vfs, p(local_vfs), "k-", alpha=0.6, linewidth=2)

ax2.set_xlabel("Local Porosity (%)", fontweight='bold')
ax2.set_ylabel(r"Local Tortuosity ($\tau$)", fontweight='bold')
ax2.set_title(f"Physical Correlation (Scale: {w_small}^3)", fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 4. From Uncertainty to Training Data

The scatter plot above reveals a clear physical trend: local geometry (porosity, pore connectivity) controls transport resistance. At small scales, the variation across crops captures a wide range of structure–property relationships.

This observation has a practical implication for machine learning. The same sub-volume sampling strategy used here to quantify uncertainty can also generate labelled training data for surrogate models. Each (morphology, tortuosity) pair is a training sample, and the spread across scales provides natural coverage of the feature space.

This idea is developed further in [Tutorial 5: Surrogate Modelling](05_surrogate_modelling.ipynb), where we train a regression model on OpenImpala-generated data. The REV analysis from this tutorial provides a useful reference: a well-calibrated surrogate should produce prediction intervals that narrow with increasing sample size, mirroring the convergence we observed above.

**Related tutorials:**
- [Tutorial 4: Multi-Phase Transport](04_multiphase_and_fields.ipynb) — Extend to composites with three or more phases.
- [Tutorial 5: Surrogate Modelling](05_surrogate_modelling.ipynb) — Use this sampling approach to train ML models.